## 1. Imports et Configuration

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from typing import Tuple, List, Union

## 2. Données d'Entrée

On considère un dictionnaire de critères avec leurs valuations respectives. 
- Les valeurs **positives** représentent les avantages ("pros")
- Les valeurs **négatives** représentent les inconvénients ("cons")
- Les valeurs **nulles** sont neutres

In [9]:
# Dictionnaire des cours et leurs contributions à la comparaison
Dictionnaire_cours = {
    "A" : 32,
    "B" : 0,
    "C" : -28,
    "D" : 36,
    "E" : 48,
    "F" : -35,
    "G" : -42
}

# Afficher les données sous forme de tableau
df = pd.DataFrame(list(Dictionnaire_cours.items()), columns=['Cours', 'Contribution x > y'])
print("Données d'entrée:")
print(df.to_string(index=False))

Données d'entrée:
Cours  Contribution x > y
    A                  32
    B                   0
    C                 -28
    D                  36
    E                  48
    F                 -35
    G                 -42


## 3. Fonctions Utilitaires

### 3.1 Identification des Pros, Cons et Neutres

In [3]:
def def_pros_cons_neutrals(dictionnaire):
    """
    Identifie les critères positifs (pros), négatifs (cons) et neutres.
    
    Args:
        dictionnaire: Dict[str, float] - critères et leurs valuations
    
    Returns:
        Tuple[List, List, List] - (pros, cons, neutrals)
    """
    pros = []
    cons = []
    neutrals = []
    
    for critere, valeur in dictionnaire.items():
        if valeur > 0:
            pros.append(critere)
        elif valeur < 0:
            cons.append(critere)
        else:
            neutrals.append(critere)
    
    return pros, cons, neutrals

# Appliquer la fonction
pros, cons, neutrals = def_pros_cons_neutrals(Dictionnaire_cours)

print("Classification des critères:")
print(f"  Pros (positifs):   {pros}")
print(f"  Cons (négatifs):   {cons}")
if neutrals:
    print(f"  Neutres:          {neutrals}")

Classification des critères:
  Pros (positifs):   ['A', 'D', 'E']
  Cons (négatifs):   ['C', 'F', 'G']
  Neutres:          ['B']


### 3.2 Calcul des Trade-offs Valides

Un **trade-off** est une paire (i,j) de type (pro, con) où la somme des contributions est positive:
$$\text{trade-off}(i, j) \text{ est valide si } v_i + v_j > 0$$

Cela signifie que l'avantage du pro compense (au moins partiellement) l'inconvénient du con.

In [4]:
def def_trade_offs(dictionnaire):
    """
    Calcule tous les trade-offs possibles (paires pro-con avec somme positive).
    
    Args:
        dictionnaire: Dict[str, float] - critères et leurs valuations
    
    Returns:
        List[List[str, str]] - liste de paires [pro, con]
    """
    pros, cons, _ = def_pros_cons_neutrals(dictionnaire)
    trade_offs = []
    
    for pro in pros:
        for con in cons:
            if dictionnaire[pro] + dictionnaire[con] > 0:
                trade_offs.append([pro, con])
    
    return trade_offs

# Calculer tous les trade-offs valides
trade_offs = def_trade_offs(Dictionnaire_cours)

print(f"\nTrade-offs valides trouvés: {len(trade_offs)}\n")
trade_offs_df = pd.DataFrame(
    [[pro, con, Dictionnaire_cours[pro], Dictionnaire_cours[con], 
      Dictionnaire_cours[pro] + Dictionnaire_cours[con]] 
     for pro, con in trade_offs],
    columns=['Pro', 'Con', 'Val(Pro)', 'Val(Con)', 'Somme']
)
print(trade_offs_df.to_string(index=False))


Trade-offs valides trouvés: 6

Pro Con  Val(Pro)  Val(Con)  Somme
  A   C        32       -28      4
  D   C        36       -28      8
  D   F        36       -35      1
  E   C        48       -28     20
  E   F        48       -35     13
  E   G        48       -42      6


## 4. Formulation du Programme Linéaire

### 4.1 Théorie de la Formulation LP

**Objectif:** Trouver une explication (1-1) de $x \succ y$, c'est-à-dire une bijection entre les cons de $y$ et un sous-ensemble de pros de $x$.

**Variables de décision:**
$$z_{ij} \in \{0, 1\} \quad \text{pour chaque trade-off } (i, j)$$
où $z_{ij} = 1$ si le trade-off $(i, j)$ est sélectionné dans l'explication.

**Fonction objectif:**
$$\text{Maximiser} \sum_{(i,j)} z_{ij}$$
(Maximiser le nombre de trade-offs sélectionnés)

**Contraintes:**
1. Chaque con doit être couvert **exactement une fois**:
$$\sum_{i \text{ tel que } (i,j) \text{ valide}} z_{ij} = 1 \quad \forall j \in \text{Cons}$$

2. Chaque pro peut être utilisé **au plus une fois**:
$$\sum_{j \text{ tel que } (i,j) \text{ valide}} z_{ij} \leq 1 \quad \forall i \in \text{Pros}$$

### 4.2 Implémentation avec Gurobi

In [5]:
def exist_explication_1_1_avec_solveur(dictionnaire) -> Tuple[bool, Union[List, str]]:
    """
    Formule et résout un programme linéaire pour trouver une explication 1-1 de x ≻ y.
    
    Args:
        dictionnaire: Dict[str, float] - critères et leurs valuations
    
    Returns:
        Tuple[bool, Union[List[List], str]]:
            - (True, explication) si une solution existe
            - (False, certificat) sinon
    """
    pros, cons, _ = def_pros_cons_neutrals(dictionnaire)
    trade_offs = def_trade_offs(dictionnaire)
    
    # Cas trivial
    if len(cons) == 0:
        return True, []  # Pas de cons à couvrir
    
    if len(trade_offs) == 0:
        certificat = f"Aucun trade-off valide n'existe. Cons à couvrir: {cons}"
        return False, certificat
    
    try:
        # === CRÉER LE MODÈLE GUROBI ===
        model = gp.Model("Explication_1_1")
        model.setParam('OutputFlag', 0)  # Désactiver l'affichage détaillé
        
        # === VARIABLES DE DÉCISION ===
        z = {}
        for i, (pro, con) in enumerate(trade_offs):
            z[i] = model.addVar(vtype=GRB.BINARY, name=f"z_{pro}_{con}")
        
        # === FONCTION OBJECTIF ===
        model.setObjective(gp.quicksum(z[i] for i in range(len(trade_offs))), GRB.MAXIMIZE)
        
        # === CONTRAINTE 1: Couverture des cons ===
        for con in cons:
            indices_con = [i for i, (pro, c) in enumerate(trade_offs) if c == con]
            if indices_con:
                model.addConstr(
                    gp.quicksum(z[i] for i in indices_con) == 1,
                    name=f"Couverture_{con}"
                )
            else:
                certificat = f"Le critère négatif '{con}' ne peut être couvert par aucun trade-off valide."
                return False, certificat
        
        # === CONTRAINTE 2: Unicité des pros ===
        for pro in pros:
            indices_pro = [i for i, (p, con) in enumerate(trade_offs) if p == pro]
            if indices_pro:
                model.addConstr(
                    gp.quicksum(z[i] for i in indices_pro) <= 1,
                    name=f"Unicite_{pro}"
                )
        
        # === RÉSOUDRE LE PROBLÈME ===
        model.optimize()
        
        # === ANALYSER LES RÉSULTATS ===
        if model.status == GRB.OPTIMAL:
            # Solution trouvée
            explication = []
            for i, (pro, con) in enumerate(trade_offs):
                if z[i].X > 0.5:  # Variable binaire = 1
                    explication.append([pro, con])
            
            # Vérifier la couverture
            cons_couverts = set(con for pro, con in explication)
            if len(cons_couverts) == len(cons):
                return True, explication
            else:
                cons_non_couverts = set(cons) - cons_couverts
                certificat = f"Impossible de couvrir tous les cons. Non couverts: {list(cons_non_couverts)}"
                return False, certificat
        
        elif model.status == GRB.INFEASIBLE:
            # === CERTIFICAT DE NON-EXISTENCE ===
            certificat = "\n🔴 CERTIFICAT DE NON-EXISTENCE\n"
            certificat += "="*50 + "\n"
            certificat += f"Le modèle est infaisable.\n"
            certificat += f"Il n'existe PAS d'explication 1-1 qui couvre tous les {len(cons)} cons.\n\n"
            certificat += f"Analyse:\n"
            certificat += f"  • Pros disponibles: {len(pros)} ({pros})\n"
            certificat += f"  • Cons à couvrir:  {len(cons)} ({cons})\n"
            certificat += f"  • Trade-offs:      {len(trade_offs)}\n\n"
            certificat += f"Raisons possibles:\n"
            certificat += f"  1. Pas assez de pros pour couvrir tous les cons\n"
            certificat += f"  2. Certains cons ne peuvent être compensés\n"
            
            # Analyser le système infaisable avec IIS
            try:
                model.computeIIS()
                iis_constrs = [c.ConstrName for c in model.getConstrs() if c.IISConstr]
                if iis_constrs:
                    certificat += f"  3. Conflits identifiés par Gurobi:\n"
                    for constr_name in iis_constrs:
                        certificat += f"     - {constr_name}\n"
            except:
                pass
            
            certificat += "\n" + "="*50
            return False, certificat
        
        else:
            certificat = f"Le solveur n'a pas convergé. Statut: {model.status}"
            return False, certificat
    
    except gp.GurobiError as e:
        return False, f"Erreur Gurobi: {str(e)}"

print("Fonction 'exist_explication_1_1_avec_solveur' définie ✓")

Fonction 'exist_explication_1_1_avec_solveur' définie ✓


## 5. Résolution et Affichage des Résultats

In [6]:
# Résoudre le problème avec Gurobi
existe, resultat = exist_explication_1_1_avec_solveur(Dictionnaire_cours)

print("\n" + "="*60)
print("RÉSULTATS DE L'OPTIMISATION")
print("="*60)

if existe:
    print("\n✅ EXPLICATION 1-1 TROUVÉE\n")
    print(f"Nombre de trade-offs sélectionnés: {len(resultat)}\n")
    
    # Afficher sous forme de tableau
    resultats_df = pd.DataFrame(
        [[pro, con, Dictionnaire_cours[pro], Dictionnaire_cours[con], 
          Dictionnaire_cours[pro] + Dictionnaire_cours[con]] 
         for pro, con in resultat],
        columns=['Pro', 'Con', 'Val(Pro)', 'Val(Con)', 'Somme']
    )
    print(resultats_df.to_string(index=False))
    
    # Vérification
    cons_couverts = sorted(set(con for pro, con in resultat))
    print(f"\nCons couverts: {cons_couverts}")
    print(f"Tous les cons couverts: {'Oui ✓' if len(cons_couverts) == len(cons) else 'Non ✗'}")
else:
    print(resultat)

Set parameter Username
Set parameter LicenseID to value 2755051
Set parameter LicenseID to value 2755051
Academic license - for non-commercial use only - expires 2026-12-15
Academic license - for non-commercial use only - expires 2026-12-15

RÉSULTATS DE L'OPTIMISATION

✅ EXPLICATION 1-1 TROUVÉE

Nombre de trade-offs sélectionnés: 3

Pro Con  Val(Pro)  Val(Con)  Somme
  A   C        32       -28      4
  D   F        36       -35      1
  E   G        48       -42      6

Cons couverts: ['C', 'F', 'G']
Tous les cons couverts: Oui ✓

RÉSULTATS DE L'OPTIMISATION

✅ EXPLICATION 1-1 TROUVÉE

Nombre de trade-offs sélectionnés: 3

Pro Con  Val(Pro)  Val(Con)  Somme
  A   C        32       -28      4
  D   F        36       -35      1
  E   G        48       -42      6

Cons couverts: ['C', 'F', 'G']
Tous les cons couverts: Oui ✓


## 6. Comparaison: Approche Heuristique vs Optimale

### 6.1 Implémentation de l'Heuristique Originale

Pour comparaison, voici l'approche heuristique gloutonne:

In [7]:
def exist_explication_1_1_heuristique(dictionnaire) -> Tuple[bool, List]:
    """
    Version heuristique gloutonne (originale).
    Sélectionne les trade-offs de manière séquentielle.
    """
    _, cons, _ = def_pros_cons_neutrals(dictionnaire)
    trade_offs = def_trade_offs(dictionnaire)
    
    union_con = []
    index_con = []
    
    for i in range(len(trade_offs)):
        if trade_offs[i][1] not in union_con:
            union_con.append(trade_offs[i][1])
            index_con.append(i)
    
    if len(union_con) < len(cons):
        return False, []
    else:
        explication = [trade_offs[i] for i in index_con]
        return True, explication

# Résoudre avec l'heuristique
existe_heur, resultat_heur = exist_explication_1_1_heuristique(Dictionnaire_cours)

print(f"Approche heuristique:")
print(f"  Solution existe: {existe_heur}")
if existe_heur:
    print(f"  Nombre de trade-offs: {len(resultat_heur)}")
    print(f"  Trade-offs: {resultat_heur}")

Approche heuristique:
  Solution existe: True
  Nombre de trade-offs: 3
  Trade-offs: [['A', 'C'], ['D', 'F'], ['E', 'G']]


## 7. Annexe: Détails Mathématiques

### Définitions Formelles

**Explication (1-1):**
Une explication (1-1) de $x \succ y$ est une bijection $f: \text{Cons}(y) \to \text{Pros}(x)$ telle que:
$$\forall j \in \text{Cons}(y): v(f(j)) + v(j) > 0$$

où $v(i)$ est la valuation du critère $i$.

**Programme Linéaire en Nombres Entiers:**
$$\begin{align}
\text{Maximiser} \quad & \sum_{(i,j)} z_{ij} \\
\text{sujet à} \quad & \sum_{i: (i,j) \text{ valide}} z_{ij} = 1 \quad \forall j \in \text{Cons} \\
& \sum_{j: (i,j) \text{ valide}} z_{ij} \leq 1 \quad \forall i \in \text{Pros} \\
& z_{ij} \in \{0, 1\}
\end{align}$$

**Certificat de Non-Existence (IIS):**
Un système de contraintes est **infaisable** s'il n'existe pas de solution satisfaisant toutes les contraintes.
Gurobi calcule un **IIS (Irreducible Inconsistent Subsystem)** qui identifie le sous-ensemble minimal de contraintes responsables de l'infaisabilité.